# 01 · Atributos de Polyvore desde el texto y la foto

Paso 3 de la etapa 3. A cada prenda de los outfits vestibles (paso 2) se le sacan del **nombre en inglés** los atributos del esquema común: tono, familia de color, patrón, material, estilo, calce, época y largo. Las reglas y los diccionarios están en `src/modelo_fly/atributos.py`.

**Entradas:** `data/processed/hito3/polyvore_items.csv` y `polyvore_outfits.csv` (de `00_polyvore_descarga`), y `color_imagen_polyvore.csv` (de `01c_color_imagenes_colab`, en Colab).
**Salida:** `data/processed/hito3/polyvore_items_atributos.csv` (no se sube a git).

**Supuestos (declararlos en el informe):**
- Si el nombre no menciona un patrón, la prenda es **lisa**; si no menciona una época, es **contemporánea**.
- Si el nombre no dice estilo, calce, material o largo, se usa la **pista de la categoría** (*Skinny Jeans* → ajustado, *Gowns* → formal, *Jeans* → denim, *Mini Skirts* → corto).
- Los **metales** (dorado, plateado) son neutros en joyería y accesorios, y acento fuerte en ropa.
- Calce y largo solo aplican a la ropa (arriba, abajo, vestido, exterior).
- **Color: manda el texto.** La foto solo rellena el tono y la familia cuando el nombre no dice el color (decisión B del 01/11/2026).

In [1]:
import sys
from pathlib import Path

import pandas as pd
from IPython.display import display

ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / "data" / "raw").is_dir()), None)
if ROOT is None:
    raise FileNotFoundError("No se encontró data/raw. Abre el notebook dentro del repositorio.")
sys.path.insert(0, str(ROOT / "src"))

from modelo_fly.atributos import SLOTS_ROPA, extraer
from modelo_fly.color_imagen import ajustar
from modelo_fly.esquema import valores_validos

PROC = ROOT / "data" / "processed" / "hito3"
SALIDA = PROC / "polyvore_items_atributos.csv"
COLOR_FOTO = PROC / "color_imagen_polyvore.csv"
ATRIBUTOS = ["tono", "familia_color", "patron", "material", "estilo", "calce", "epoca", "largo"]

items = pd.read_csv(PROC / "polyvore_items.csv", dtype=str, keep_default_na=False)
outfits = pd.read_csv(PROC / "polyvore_outfits.csv", dtype=str, keep_default_na=False)
validos = set(outfits.loc[outfits["valido"] == "True", "set_id"])
items = items[items["set_id"].isin(validos) & (items["slot"] != "excluido")].reset_index(drop=True)
print("Ítems de moda en outfits vestibles:", len(items))

Ítems de moda en outfits vestibles: 81372


## 1. Extracción

In [2]:
attrs = pd.DataFrame([
    extraer(n, c, s, l) for n, c, s, l in zip(items["name"], items["categoria"], items["slot"], items["largo_categoria"])
])
items = pd.concat([items, attrs], axis=1)

for a in ATRIBUTOS:
    fuera = set(items[a]) - valores_validos(a)
    if fuera:
        raise ValueError(f"{a}: valores fuera del esquema {sorted(fuera)}")
print("Todos los valores están en el vocabulario del esquema.")

ejemplos = items.sample(12, random_state=3)[["slot", "name", *ATRIBUTOS]]
display(ejemplos)

Todos los valores están en el vocabulario del esquema.


,slot,name,tono,familia_color,patron,material,estilo,calce,epoca,largo
59185,bolso,valentino rockstud leather shoulder bag,desconocido,desconocido,liso,cuero,desconocido,no_aplica,contemporaneo,no_aplica
43577,vestido,true lace sweetheart skater dress,desconocido,desconocido,liso,encaje,desconocido,desconocido,contemporaneo,desconocido
4464,abajo,gucci butterfly pocket jeans,desconocido,desconocido,liso,denim,desconocido,desconocido,contemporaneo,desconocido
18060,accesorio,jimmy choo metal sunglasses,desconocido,desconocido,liso,desconocido,desconocido,no_aplica,contemporaneo,no_aplica
69317,exterior,topshop front boyfriend coat,desconocido,desconocido,liso,desconocido,desconocido,oversize,contemporaneo,desconocido
66832,vestido,boat neck flare sleeve pleated dress,desconocido,desconocido,liso,desconocido,desconocido,desconocido,contemporaneo,desconocido
14916,calzado,pre-owned aquazzura beverly lace-up sandals,desconocido,desconocido,liso,encaje,desconocido,no_aplica,contemporaneo,no_aplica
62383,joyeria,bvlgari 18kt diamond bangle,desconocido,desconocido,liso,desconocido,desconocido,no_aplica,contemporaneo,no_aplica
63582,arriba,seconds summer boxy tank design 5sos luke mich...,desconocido,desconocido,liso,desconocido,desconocido,oversize,contemporaneo,desconocido
56283,accesorio,ray-ban clubmaster classic sunglasses,desconocido,desconocido,liso,desconocido,desconocido,no_aplica,contemporaneo,no_aplica


## 2. Color desde la foto

El CSV de Colab trae el color dominante de cada foto, con la clave de la copia de imágenes (*Re-PolyVore*). Esa copia guarda cada producto una sola vez, así que la unión se hace por **producto** (`tid`) y no por `item_key`. Después se aplica `ajustar` (gris → segundo tono, blanco o negro; metales en joyería).

Primero se mide cuánto coincide la foto con el texto donde el nombre sí dice el color. Los umbrales del ajuste se eligieron con la partición oficial *train*; la cifra honesta es la de *valid + test*.

In [3]:
color = pd.read_csv(COLOR_FOTO, dtype={"item_key": str}, keep_default_na=False, na_values={"frac_tono2": ""})
print("Fotos procesadas:", len(color), "| con error:", color["error"].ne("").sum())

todos = pd.read_csv(PROC / "polyvore_items.csv", dtype=str, usecols=["item_key", "tid"])
color = color.merge(todos, on="item_key").drop_duplicates("tid").drop(columns=["item_key", "carpeta", "error"])
items = items.merge(color, on="tid", how="left")
con_foto = items["tono_img"].notna()
items["tono_foto"], items["familia_foto"] = "", ""
items.loc[con_foto, "tono_foto"], items.loc[con_foto, "familia_foto"] = ajustar(items[con_foto], items.loc[con_foto, "slot"])
print(f"Prendas con foto: {con_foto.sum()} de {len(items)} ({con_foto.mean():.1%})")
display(items.groupby("slot")["tono_img"].apply(lambda s: round(100 * s.notna().mean(), 1)).to_frame("% con foto"))

Fotos procesadas: 126928 | con error: 0


Prendas con foto: 60486 de 81372 (74.3%)


,% con foto
slot,
abajo,70.9
accesorio,72.3
arriba,84.3
bolso,85.6
calzado,67.4
exterior,78.5
joyeria,62.8
vestido,84.5


In [4]:
ambos = items[con_foto & items["tono"].ne("desconocido")]
grupo = ambos["particion_oficial"].eq("train").map({True: "train (calibración)", False: "valid + test"})
acuerdo = pd.DataFrame({
    "prendas": grupo.value_counts(),
    "% mismo tono (sin ajuste)": (ambos["tono"] == ambos["tono_img"]).groupby(grupo).mean().mul(100).round(1),
    "% mismo tono": (ambos["tono"] == ambos["tono_foto"]).groupby(grupo).mean().mul(100).round(1),
    "% misma familia": (ambos["familia_color"] == ambos["familia_foto"]).groupby(grupo).mean().mul(100).round(1),
})
display(acuerdo)
print("Acuerdo por slot (todas las particiones):")
display(ambos.groupby("slot").apply(lambda d: pd.Series({
    "prendas": len(d),
    "% mismo tono": round(100 * (d["tono"] == d["tono_foto"]).mean(), 1),
    "% misma familia": round(100 * (d["familia_color"] == d["familia_foto"]).mean(), 1)})))
print("Familia del texto (filas) contra familia de la foto (columnas):")
display(pd.crosstab(ambos["familia_color"], ambos["familia_foto"]))

,prendas,% mismo tono (sin ajuste),% mismo tono,% misma familia
particion_oficial,,,,
train (calibración),15486,48.1,61.1,80.7
valid + test,2377,43.7,57.0,76.9


Acuerdo por slot (todas las particiones):


,prendas,% mismo tono,% misma familia
slot,,,
abajo,2094.0,69.8,77.2
accesorio,2233.0,46.6,84.1
arriba,2203.0,64.1,80.8
bolso,2624.0,64.6,82.8
calzado,2810.0,65.9,84.9
exterior,1233.0,60.9,77.3
joyeria,3854.0,54.4,76.5
vestido,812.0,63.4,72.9


Familia del texto (filas) contra familia de la foto (columnas):


familia_foto,fuerte,neutro,suave
familia_color,,,
fuerte,2094,2245,208
neutro,525,12180,44
suave,79,435,53


In [5]:
rellenar = con_foto & items["tono"].eq("desconocido")
items.loc[rellenar, "tono"] = items.loc[rellenar, "tono_foto"]
items.loc[rellenar, "familia_color"] = items.loc[rellenar, "familia_foto"]
items.loc[rellenar, ["origen_tono", "origen_familia_color"]] = "foto"
print(f"Tono rellenado desde la foto: {rellenar.sum()} prendas")
for a in ["tono", "familia_color"]:
    fuera = set(items[a]) - valores_validos(a)
    if fuera:
        raise ValueError(f"{a}: valores fuera del esquema {sorted(fuera)}")

Tono rellenado desde la foto: 42623 prendas

## 3. Cobertura

Qué parte de las prendas tiene un valor leído del texto, de la categoría o de la foto, cuántas quedan por supuesto (liso, contemporáneo) y cuántas quedan desconocidas. Calce y largo se miden solo sobre la ropa.

In [6]:
def fuente(origen):
    if origen in ("-", "sin nombre"):
        return "desconocido"
    if origen == "supuesto":
        return "supuesto"
    if origen in ("regla", "foto"):
        return origen
    if origen == "no aplica al slot":
        return "no aplica"
    return "categoria" if origen.startswith("categoria:") else "texto"

ropa = items["slot"].isin(SLOTS_ROPA)
filas = []
for a in ATRIBUTOS:
    base = items[ropa] if a in ("calce", "largo") else items
    f = base[f"origen_{a}"].map(fuente)
    if a == "familia_color":
        f = base["origen_familia_color"].where(base["familia_color"].ne("desconocido"), "desconocido")
    cuenta = f.value_counts(normalize=True).mul(100).round(1)
    filas.append({"atributo": a, "base": "ropa" if a in ("calce", "largo") else "todas", **cuenta.to_dict()})
cobertura = pd.DataFrame(filas).set_index("atributo").fillna(0)
cobertura = cobertura[[c for c in ["base", "texto", "categoria", "foto", "regla", "supuesto", "desconocido"] if c in cobertura]]
display(cobertura)

print("Distribución de valores:")
for a in ATRIBUTOS:
    print(f"  {a}:", items[a].value_counts().head(8).to_dict())

,base,texto,categoria,foto,regla,supuesto,desconocido
atributo,,,,,,,
tono,todas,30.3,0.0,52.4,0.0,0.0,17.3
familia_color,todas,0.0,0.0,52.4,30.3,0.0,17.3
patron,todas,10.1,0.0,0.0,0.0,89.2,0.7
material,todas,32.2,0.1,0.0,0.0,0.0,67.7
estilo,todas,5.3,12.7,0.0,0.0,0.0,82.0
calce,ropa,14.1,3.1,0.0,0.0,0.0,82.8
epoca,todas,2.2,0.0,0.0,0.0,97.1,0.7
largo,ropa,21.3,6.2,0.0,0.0,0.0,72.5


Distribución de valores:
  tono: {'negro': 19870, 'desconocido': 14097, 'blanco': 7316, 'beige': 5377, 'azul': 5348, 'dorado': 5157, 'gris': 4330, 'rojo': 3993}
  familia_color: {'neutro': 52211, 'desconocido': 14097, 'fuerte': 13559, 'suave': 1505}
  patron: {'liso': 72579, 'floral': 2295, 'estampado': 2187, 'rayas': 1340, 'cuadros': 854, 'animal': 720, 'desconocido': 542, 'grafico': 537}
  material: {'desconocido': 55090, 'cuero': 8236, 'denim': 5882, 'gamuza': 2548, 'encaje': 1956, 'lana': 1951, 'algodon': 1927, 'punto': 1583}
  estilo: {'desconocido': 66722, 'formal': 8345, 'athleisure': 3500, 'vintage': 1660, 'casual': 801, 'deportivo': 344}
  calce: {'no_aplica': 49043, 'desconocido': 26767, 'ajustado': 3532, 'oversize': 1739, 'recto': 291}
  epoca: {'contemporaneo': 79026, 'vintage_generico': 1665, 'desconocido': 542, '1990s': 73, '1980s': 39, '1970s': 16, '1950s': 11}
  largo: {'no_aplica': 49043, 'desconocido': 23435, 'corto': 5960, 'midi': 2041, 'largo': 893}


In [7]:
print("Tono conocido por slot (%):")
display(items.groupby("slot")["tono"].apply(lambda s: round(100 * s.ne("desconocido").mean(), 1)).to_frame("% con color"))

outfit_color = items.groupby("set_id")["tono"].apply(lambda s: s.ne("desconocido").mean())
print(f"Outfits donde se conoce el color de todas las prendas: {(outfit_color == 1).mean():.1%}")
print(f"Outfits donde se conoce el color de al menos la mitad: {(outfit_color >= 0.5).mean():.1%}")
print(f"Outfits sin ningún color conocido: {(outfit_color == 0).mean():.1%}")

Tono conocido por slot (%):


,% con color
slot,
abajo,78.4
accesorio,79.7
arriba,89.8
bolso,90.1
calzado,77.3
exterior,85.0
joyeria,78.7
vestido,89.0


Outfits donde se conoce el color de todas las prendas: 40.0%
Outfits donde se conoce el color de al menos la mitad: 95.7%
Outfits sin ningún color conocido: 0.3%


## 4. Salida

In [8]:
cols = ["item_key", "set_id", "index", "tid", "particion_oficial", "categoryid", "categoria", "slot", "genero",
        "name", "texto", *ATRIBUTOS, *[f"origen_{a}" for a in ATRIBUTOS],
        "tono_foto", "familia_foto", "frac_tono", "frac_prenda"]
items[cols].to_csv(SALIDA, index=False, encoding="utf-8")
print("Guardado:", SALIDA.relative_to(ROOT), items[cols].shape)

Guardado: data\processed\hito3\polyvore_items_atributos.csv (81372, 31)
